<a href="https://colab.research.google.com/github/Huzaifa206/arch-technologies-internship/blob/main/Task%204/Task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

!pip install transformers librosa soundfile gTTS

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-0gmrvzjy/unsloth_2019357df81246b79d0c64e6f9494278
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-0gmrvzjy/unsloth_2019357df81246b79d0c64e6f9494278
  Resolved https://github.com/unslothai/unsloth.git to commit ac2daf8b7a4df093d597a9d1e5e6ffdb5cb0804c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.0 MB/s eta 0:00:0

In [2]:
import torch
import gc
from transformers import pipeline
from unsloth import FastLanguageModel
from gtts import gTTS
import IPython.display as ipd

# Clear any lingering memory
torch.cuda.empty_cache()
gc.collect()

print("Libraries imported and VRAM cleared!")

/tmp/ipykernel_3692/3210098035.py:4: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Libraries imported and VRAM cleared!


In [3]:
print("Loading OpenAI Whisper (ASR)...")

# Load whisper-small onto the GPU in 16-bit precision
whisper_asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device="cuda",
    torch_dtype=torch.float16
)

print("Whisper loaded successfully!")

Loading OpenAI Whisper (ASR)...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Whisper loaded successfully!


In [5]:
print("Loading Unsloth Qwen 2.5 4-bit (Reasoning LLM)...")
max_seq_length = 2048

# Load the dynamic 4-bit model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Enable 2x faster generation speed
FastLanguageModel.for_inference(model)

print("Qwen Reasoning Model loaded successfully!")

Loading Unsloth Qwen 2.5 4-bit (Reasoning LLM)...
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Qwen Reasoning Model loaded successfully!


In [6]:
audio_file_path = "sample_query.mp3"
test_spoken_query = "A patient comes in with a severe headache, stiff neck, and a high fever. They are also sensitive to bright lights. What is the most likely diagnosis, and what test should be ordered immediately?"

print(f"Generating test audio file...")
tts = gTTS(test_spoken_query, lang='en')
tts.save(audio_file_path)

# This displays an audio player in Colab so you can actually click play and hear it!
ipd.display(ipd.Audio(audio_file_path))

Generating test audio file...


In [10]:
def speech_to_reasoning(audio_path):
    print("\n" + "="*50)
    print("🎙️ STEP 1: TRANSCRIBING AUDIO...")

    # 1. Pass the audio file to Whisper
    transcription_result = whisper_asr(audio_path)
    transcribed_text = transcription_result["text"].strip()
    print(f"Transcription: \"{transcribed_text}\"")

    print("-" * 50)
    print("🧠 STEP 2: REASONING & GENERATING ANSWER...")

    # 2. Define the reasoning prompt template
    messages = [
        {"role": "system", "content": "You are a logical reasoning medical assistant. Always break down your thought process step-by-step before providing a final answer."},
        {"role": "user", "content": transcribed_text}
    ]

    # Format the prompt using Qwen's specific chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize and push to GPU
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # 3. Generate the response
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=500,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # 4. Decode and clean the output
    raw_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Split the raw output to isolate just the AI's reply (everything after 'assistant')
    final_answer = raw_output.split("assistant\n")[-1]

    print("\n" + final_answer.strip())
    print("="*50)

In [11]:
speech_to_reasoning(audio_file_path)


🎙️ STEP 1: TRANSCRIBING AUDIO...


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

Transcription: "A patient comes in with a severe headache, stiff neck, and a high fever. They are also sensitive to bright lights. What is the most likely diagnosis? And what test should be ordered immediately?"
--------------------------------------------------
🧠 STEP 2: REASONING & GENERATING ANSWER...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


Let's break this down step-by-step to determine the most likely diagnosis and the appropriate immediate test:

1. **Symptoms Analysis**:
   - **Severe headache**: This can be associated with various conditions but often points towards a more serious underlying issue.
   - **Stiff neck**: This is a classic sign of meningitis and indicates inflammation of the meninges (the protective membranes covering the brain and spinal cord).
   - **High fever**: Fever is a common symptom of infection, including bacterial or viral infections.
   - **Sensitivity to bright lights (photophobia)**: This is another common symptom of meningitis.

2. **Clinical Picture**:
   - The combination of these symptoms strongly suggests a potential case of meningitis, which can be caused by bacteria or viruses.
   - Bacterial meningitis is particularly concerning due to its rapid progression and severity.

3. **Likely Diagnosis**:
   - Given the symptoms, the most likely diagnosis is bacterial meningitis, especiall